In [1]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data_dir = "/datasets/slow/LUNA25"
workspace_root = "/models/hiera-luna25"
out_dir = Path(f"{workspace_root}/figures/diagnostics")
out_dir.mkdir(parents=True, exist_ok=True)

df_annotations = pd.read_csv(f"{data_dir}/LUNA25_Public_Training_Development_Data.csv")
df_volumes = pd.read_csv(f"{workspace_root}/mask_volumes.csv")
df_volumes["SeriesInstanceUID"] = df_volumes["filename"].str.replace(".nii.gz", "", regex=False)
df_predictions = pd.read_csv(f"{workspace_root}/results/finetune-hiera-3D-20260814/all_folds_predictions.csv")
df_radiomics = pd.read_csv(f"{data_dir}/additional-annotations/csv-annotations/radiomics.csv",
                           usecols=["AnnotationID", "original_shape_Sphericity", "original_firstorder_Entropy"])
df = pd.merge(df_annotations, df_volumes.drop(columns="filename"), on="SeriesInstanceUID", how="left")
df = pd.merge(df, df_predictions, on="AnnotationID", how="inner")
assert (df["label_x"] == df["label_y"]).all(), "Labels in annotations and predictions are not the same!"
df = df.drop(columns="label_y").rename(columns={"label_x": "label"})
df = pd.merge(df, df_radiomics, on="AnnotationID", how="left")
df = df.rename(columns={"original_shape_Sphericity": "sphericity", "original_firstorder_Entropy": "entropy"})
n_all = len(df)
df = df.dropna(subset=["volume_mm3", "sphericity", "entropy"]).reset_index(drop=True)
print(f"{n_all - len(df)} of {n_all} rows have no radiomics/volume and are dropped; {len(df)} left")

df["log_volume"] = np.log1p(df["volume_mm3"])
df["hiera_logit"] = logit(df["probability"].clip(1e-6, 1 - 1e-6))
df["diameter_mm"] = 2 * ((3 * df["volume_mm3"]) / (4 * math.pi)) ** (1 / 3)

197 of 4968 rows have no radiomics/volume and are dropped; 4771 left


In [2]:
# https://github.com/DIAGNijmegen/luna25-evaluation-public
def calculate_auc(y_true, y_pred):
    auc = roc_auc_score(y_true, y_pred)
    rng = np.random.RandomState(seed=42)
    bootstrapped_aucs = []
    for _ in range(1000):
        indices = rng.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true.iloc[indices])) < 2:
            continue
        bootstrapped_aucs.append(roc_auc_score(y_true.iloc[indices], y_pred.iloc[indices]))
    return {"auc": auc, "ci_lower": np.percentile(bootstrapped_aucs, 2.5), "ci_upper": np.percentile(bootstrapped_aucs, 97.5)}

# out-of-fold logistic regression on the same folds as Hiera
feature_sets = {
    "volume": ["log_volume"],
    "volume+shape+texture": ["log_volume", "sphericity", "entropy"],
    "volume+hiera": ["log_volume", "hiera_logit"],
}
for name, cols in feature_sets.items():
    df[name] = np.nan
    for k in range(1, 6):
        train, test = df["fold"] != k, df["fold"] == k
        model = make_pipeline(StandardScaler(), LogisticRegression()).fit(df.loc[train, cols], df.loc[train, "label"])
        df.loc[test, name] = model.predict_proba(df.loc[test, cols])[:, 1]
df["hiera"] = df["probability"]
scores = ["hiera", "volume", "volume+shape+texture", "volume+hiera"]

rows = {}
for s in scores:
    fold_aucs = [roc_auc_score(g["label"], g[s]) for _, g in df.groupby("fold")]
    pooled = calculate_auc(df["label"], df[s])
    rows[s] = {**{f"fold {k}": a for k, a in enumerate(fold_aucs, 1)}, "mean": np.mean(fold_aucs), "std": np.std(fold_aucs),
               "pooled": pooled["auc"], "ci_low": pooled["ci_lower"], "ci_high": pooled["ci_upper"]}
table1 = pd.DataFrame(rows).T.round(4)
display(table1)
delta = table1.loc["volume+hiera"] - table1.loc["volume"]
print(f"AUC(volume+hiera) - AUC(volume): pooled {delta['pooled']:.4f}, per-fold mean {delta['mean']:.4f}")
print(f"check: volume LR pooled AUC {table1.loc['volume', 'pooled']:.4f} vs roc_auc_score(label, volume_mm3) {roc_auc_score(df['label'], df['volume_mm3']):.4f}")

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std,pooled,ci_low,ci_high
hiera,0.9065,0.8882,0.9125,0.8835,0.8996,0.8981,0.0109,0.8949,0.8808,0.9080
volume,0.7246,0.6561,0.6843,0.7218,0.7061,0.6986,0.0256,0.6961,0.6709,0.7213
volume+shape+texture,0.7419,0.6828,0.7111,0.7594,0.7398,0.7270,0.0270,0.7265,0.7017,0.7493
volume+hiera,0.9060,0.8887,0.9128,0.8837,0.8995,0.8981,0.0107,0.8926,0.8781,0.9059


AUC(volume+hiera) - AUC(volume): pooled 0.1965, per-fold mean 0.1995
check: volume LR pooled AUC 0.6961 vs roc_auc_score(label, volume_mm3) 0.6976


In [3]:
df["size_bin"] = pd.cut(df["diameter_mm"], [0, 6, 8, 15, np.inf])
def bin_row(g):
    row = {"n": len(g), "n_malignant": int(g["label"].sum())}
    for s in scores:
        row[s] = roc_auc_score(g["label"], g[s]) if g["label"].nunique() == 2 else np.nan
    return pd.Series(row)
table2 = df.groupby("size_bin", observed=False).apply(bin_row).round(4)
display(table2)
print(f"sum of n in Table 2: {int(table2['n'].sum())} (rows used: {len(df)})")

table1.to_csv(out_dir / "auc_by_fold.csv")
table2.to_csv(out_dir / "auc_by_size_bin.csv")

,n,n_malignant,hiera,volume,volume+shape+texture,volume+hiera
size_bin,,,,,,
"(0.0, 6.0]",1044.0,40.0,0.9078,0.6231,0.6418,0.9037
"(6.0, 8.0]",1157.0,67.0,0.8699,0.5458,0.6844,0.8679
"(8.0, 15.0]",1863.0,174.0,0.8778,0.5781,0.6758,0.8741
"(15.0, inf]",707.0,165.0,0.8633,0.5838,0.5449,0.8604


sum of n in Table 2: 4771 (rows used: 4771)


In [4]:
# AUC and operating points per NLST diameter bin for every run, on the nodules every run predicted (no radiomics filter)
from scipy.stats import binomtest

runs = {  # add new experiments here
    "baseline": "finetune-hiera-3D-20260814",
    "no mixup": "finetune-hiera-nomixup-3D-20261022",
    "z 1.6 mm": "finetune-hiera-z1.6mm-3D-20261023",
    "depth 32": "finetune-hiera-d32-3D-20261024",
    "depth 32 + zoom": "finetune-hiera-d32-zoom-3D-20261024",
    "xy FOV 25 mm": "finetune-hiera-xy25mm-3D-20261024",
}
preds = {name: pd.concat([pd.read_csv(f"{workspace_root}/results/{d}/fold_{k}/predictions.csv") for k in range(1, 6)])
         for name, d in runs.items()}
nlst = pd.read_csv(f"{data_dir}/additional-annotations/csv-annotations/lesion_nlst_diameters_nolocal.csv",
                   usecols=["AnnotationID", "nlst_diameter_mm"]).dropna()
assert not nlst["AnnotationID"].duplicated().any()
common = set.intersection(*(set(p["AnnotationID"]) for p in preds.values()))
print({name: len(p) for name, p in preds.items()}, f"common {len(common)}, with NLST diameter {len(common & set(nlst['AnnotationID']))}")

def wilson(k, n):
    ci = binomtest(k, n).proportion_ci(method="wilson")
    return k / n, ci.low, ci.high

def nlst_bin_row(g, thr_spec, thr_sens):
    mal, ben = g.loc[g["label"] == 1, "probability"], g.loc[g["label"] == 0, "probability"]
    auc = calculate_auc(g["label"], g["probability"])
    sens = wilson(int((mal > thr_spec).sum()), len(mal))
    spec = wilson(int((ben <= thr_sens).sum()), len(ben))
    return pd.Series({"n": len(g), "n_malignant": len(mal),
                      "auc": auc["auc"], "auc_ci_low": auc["ci_lower"], "auc_ci_high": auc["ci_upper"],
                      "auc_diameter_only": roc_auc_score(g["label"], g["nlst_diameter_mm"]),
                      "sens@95spec": sens[0], "sens_ci_low": sens[1], "sens_ci_high": sens[2],
                      "spec@95sens": spec[0], "spec_ci_low": spec[1], "spec_ci_high": spec[2]})

tables = []
for name, p in preds.items():
    assert not p["AnnotationID"].duplicated().any()
    dn = p[p["AnnotationID"].isin(common)].merge(nlst, on="AnnotationID")
    # thresholds from each run's own pooled predictions, so every bin is read at that run's operating point
    thr_spec = dn.loc[dn["label"] == 0, "probability"].quantile(0.95)  # 95% specificity
    thr_sens = dn.loc[dn["label"] == 1, "probability"].quantile(0.05)  # 95% sensitivity
    dn["nlst_bin"] = pd.cut(dn["nlst_diameter_mm"], [0, 6, 8, 12.5, 20, np.inf])
    t = pd.concat([dn.groupby("nlst_bin", observed=True).apply(nlst_bin_row, thr_spec, thr_sens).rename(index=str),
                   nlst_bin_row(dn, thr_spec, thr_sens).to_frame("all").T])
    tables.append(t.rename_axis("nlst_bin").reset_index().assign(run=name))
table3 = pd.concat(tables).set_index(["run", "nlst_bin"]).round(4)
pd.set_option("display.width", 250, "display.max_columns", 20, "display.max_rows", 100)
display(table3)
table3.to_csv(out_dir / "auc_by_nlst_diameter.csv")

{'baseline': 4968, 'no mixup': 4968, 'z 1.6 mm': 4968, 'depth 32': 4968, 'depth 32 + zoom': 4968, 'xy FOV 25 mm': 4968} common 4968, with NLST diameter 4882


n  n_malignant     auc  auc_ci_low  auc_ci_high  auc_diameter_only  sens@95spec  sens_ci_low  sens_ci_high  spec@95sens  spec_ci_low  spec_ci_high
run             nlst_bin                                                                                                                                                             
baseline        (0.0, 6.0]    2355.0         49.0  0.8908      0.8417       0.9346             0.6763       0.0612       0.0210        0.1652       0.8287       0.8128        0.8435
                (6.0, 8.0]    1011.0         63.0  0.8154      0.7588       0.8702             0.5276       0.1905       0.1125        0.3041       0.5886       0.5570        0.6195
                (8.0, 12.5]    768.0        116.0  0.7788      0.7363       0.8157             0.5251       0.3362       0.2567        0.4262       0.3773       0.3409        0.4151
                (12.5, 20.0]   460.0        113.0  0.8308      0.7914       0.8684             0.5755       0.6283       0.5364        0.7118       0.3256       0.2785        0.3766
                (20.0, inf]    288.0        105.0  0.8067      0.7526       0.8526             0.5390       0.7714       0.6824        0.8413       0.2131       0.1600        0.2780
                all           4882.0        446.0  0.8949      0.8805       0.9086             0.8043       0.4619       0.4161        0.5083       0.6463       0.6321        0.6602
no mixup        (0.0, 6.0]    2355.0         49.0  0.8218      0.7571       0.8879             0.6763       0.1429       0.0710        0.2667       0.6982       0.6791        0.7166
                (6.0, 8.0]    1011.0         63.0  0.7933      0.7321       0.8492             0.5276       0.2381       0.1499        0.3564       0.4230       0.3919        0.4547
                (8.0, 12.5]    768.0        116.0  0.7890      0.7497       0.8276             0.5251       0.3793       0.2962        0.4701       0.2929       0.2593        0.3290
                (12.5, 20.0]   460.0        113.0  0.8319      0.7897       0.8697             0.5755       0.6726       0.5816        0.7522       0.2277       0.1867        0.2746
                (20.0, inf]    288.0        105.0  0.8137      0.7606       0.8615             0.5390       0.7048       0.6115        0.7835       0.1803       0.1314        0.2424
                all           4882.0        446.0  0.8836      0.8679       0.9002             0.8043       0.4843       0.4383        0.5306       0.5216       0.5069        0.5363
z 1.6 mm        (0.0, 6.0]    2355.0         49.0  0.8890      0.8439       0.9288             0.6763       0.0204       0.0036        0.1069       0.8092       0.7926        0.8247
                (6.0, 8.0]    1011.0         63.0  0.7987      0.7418       0.8528             0.5276       0.2540       0.1628        0.3734       0.5285       0.4967        0.5601
                (8.0, 12.5]    768.0        116.0  0.7735      0.7329       0.8111             0.5251       0.3534       0.2724        0.4439       0.3206       0.2859        0.3573
                (12.5, 20.0]   460.0        113.0  0.7944      0.7519       0.8359             0.5755       0.5752       0.4831        0.6624       0.1614       0.1264        0.2038
                (20.0, inf]    288.0        105.0  0.7969      0.7460       0.8435             0.5390       0.6762       0.5818        0.7581       0.1311       0.0897        0.1877
                all           4882.0        446.0  0.8877      0.8730       0.9020             0.8043       0.4350       0.3897        0.4813       0.5987       0.5842        0.6131
depth 32        (0.0, 6.0]    2355.0         49.0  0.8964      0.8561       0.9315             0.6763       0.0204       0.0036        0.1069       0.8374       0.8218        0.8519
                (6.0, 8.0]    1011.0         63.0  0.8136      0.7600       0.8616             0.5276       0.2857       0.1890        0.4070       0.5949       0.5634        0.6257
                (8.0, 12.5]    